# XGBoost: speed and missing values
Training time of classic gradient boosting against XGBoost on the CPU, scikit-learn's histogram-based
gradient boosting and LightGBM; XGBoost on data with missing values; equal-width against quantile bins.

In [1]:
import os

# one processor core in every library, so the comparison is about the algorithms, not the number of cores
os.environ["OMP_NUM_THREADS"] = "1"

import time

import lightgbm as lgb
import numpy as np
import pandas as pd
import xgboost as xgb
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, HistGradientBoostingClassifier
from sklearn.model_selection import cross_val_score, train_test_split
from xgboost import XGBClassifier

print("xgboost", xgb.__version__, "| lightgbm", lgb.__version__)

xgboost 3.4.2 | lightgbm 4.7.0


## 1. Training time on 10,000 observations and 200 features (one core, processor time)
Processor time (`time.process_time`) counts only the time this process spends computing, so it is much less affected by other programs running on the machine than wall-clock time.

In [2]:
# a synthetic dataset: 10,000 rows, 200 columns (20 of them informative), two classes
X, y = make_classification(n_samples=10_000, n_features=200, n_informative=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# every model: 100 trees, learning rate 0.1, trees of depth 3, so the comparison is about speed only
models = {
    # classic gradient boosting: every split value tried exactly, one core (it has no n_jobs)
    "sklearn GradientBoosting": GradientBoostingClassifier(n_estimators=100, max_depth=3, random_state=42),
    # XGBoost: histogram bins, one core
    "XGBoost": XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, tree_method="hist",
                             device="cpu", n_jobs=1, random_state=42),
    # scikit-learn's histogram-based gradient boosting (modelled on LightGBM)
    "sklearn HistGradientBoosting": HistGradientBoostingClassifier(max_iter=100, max_depth=3,
                                                                   early_stopping=False, random_state=42),
    # LightGBM
    "LightGBM": lgb.LGBMClassifier(n_estimators=100, max_depth=3, num_leaves=8, learning_rate=0.1,
                                   n_jobs=1, random_state=42, verbose=-1),
}
rows = []
for name, model in models.items():
    model.fit(X_train[:500], y_train[:500])          # warm-up (the first call loads libraries)
    start = time.process_time()
    model.fit(X_train, y_train)                      # processor time of the real training only
    seconds = time.process_time() - start
    rows.append((name, round(seconds, 2), round(model.score(X_test, y_test), 3)))
timings = pd.DataFrame(rows, columns=["model", "seconds", "test accuracy"])
timings.to_csv("data/timings.csv", index=False)       # the Note's bar chart reads this file
timings

,model,seconds,test accuracy
0,sklearn GradientBoosting,59.87,0.922
1,XGBoost,1.88,0.921
2,sklearn HistGradientBoosting,1.21,0.916
3,LightGBM,1.13,0.918


## 2. Missing values: the Titanic ages

In [3]:
# 891 passengers; Age is missing for 177 of them
df = pd.read_csv("data/titanic_train.csv")
df["Sex"] = (df["Sex"] == "female").astype(int)      # 1 = female, 0 = male
X = df[["Pclass", "Sex", "Age", "Fare"]]
y = df["Survived"]
print(X.isna().sum())

Pclass      0
Sex         0
Age       177
Fare        0
dtype: int64


In [4]:
# classic gradient boosting refuses missing values
try:
    GradientBoostingClassifier().fit(X, y)
except ValueError as e:
    print("ValueError:", str(e).splitlines()[0])

ValueError: Input X contains NaN.


In [5]:
# XGBoost trains on them as they are (np.nan is its default marker for 'missing')
xgb_model = XGBClassifier(n_estimators=100, max_depth=3, learning_rate=0.1, missing=np.nan, random_state=42)
print("5-fold CV accuracy, NaN kept:     ", cross_val_score(xgb_model, X, y, cv=5).mean().round(3))

# for comparison: fill the missing ages with the median first
X_filled = X.fillna({"Age": X["Age"].median()})
print("5-fold CV accuracy, median-filled:", cross_val_score(xgb_model, X_filled, y, cv=5).mean().round(3))

5-fold CV accuracy, NaN kept:      0.828


5-fold CV accuracy, median-filled: 0.834


In [6]:
# the learned default direction: in the first tree, where do passengers with no Age go?
xgb_model.fit(X, y)
tree0 = xgb_model.get_booster().trees_to_dataframe().query("Tree == 0")
age_splits = tree0[tree0["Feature"] == "Age"][["ID", "Split", "Yes", "No", "Missing"]]
age_splits["missing goes"] = np.where(age_splits["Missing"] == age_splits["Yes"], "left (Age < split)",
                                      "right (Age >= split)")
age_splits

,ID,Split,Yes,No,Missing,missing goes
1,0-1,13.0,0-3,0-4,0-4,right (Age >= split)


## 3. Equal-width bins and quantile bins on a skewed column

In [7]:
fare = df["Fare"]
print(fare.describe().round(1))

# 8 bins of equal width
uniform_edges = np.linspace(fare.min(), fare.max(), 9)
# 8 bins with about the same number of passengers each (quantiles 0, 1/8, ..., 1)
quantile_edges = np.quantile(fare, np.linspace(0, 1, 9))

print("uniform edges: ", uniform_edges.round(1))
print("rows per bin:  ", np.histogram(fare, uniform_edges)[0])
print("quantile edges:", quantile_edges.round(1))
print("rows per bin:  ", np.histogram(fare, quantile_edges)[0])

count    891.0
mean      32.2
std       49.7
min        0.0
25%        7.9
50%       14.5
75%       31.0
max      512.3
Name: Fare, dtype: float64
uniform edges:  [  0.   64.  128.1 192.1 256.2 320.2 384.2 448.3 512.3]
rows per bin:   [773  80  18  11   6   0   0   3]
quantile edges: [  0.    7.8   7.9   9.8  14.5  24.5  31.   69.5 512.3]
rows per bin:   [106 117 111 106 117 109 113 112]
